# Headline Cleanup Pipeline

End-to-end pipeline for filtering the WSJ headline corpus to financially
relevant articles before feeding into the main news encoder pipeline.

**Three-stage filter:**
1. **Structural filter** — remove empty, too-short, or malformed headlines
2. **Blocklist filter** — remove headlines containing explicit non-financial terms
3. **Anchor similarity filter** — remove headlines semantically distant from
   financial anchor phrases using frozen FinBERT embeddings

**Output:** a filtered index array saved to disk. The main pipeline loads this
and applies it when aggregating wave embeddings — no re-encoding required.

**Prerequisites:** run the main pipeline notebook first so `model`, `tokenizer`,
`article_embeddings`, and `news_df` are in memory. Or set `STANDALONE = True`
to load from disk.

## 1. Configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hashlib, os, re, warnings
from tqdm import tqdm
warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR  = Path("./data")
CACHE_DIR = Path("/hpc/dctrl/ah620/storgae")
OUTPUT_DIR = Path("./output")
for d in [CACHE_DIR, OUTPUT_DIR]:
    d.mkdir(exist_ok=True, parents=True)

NEWS_PARQUETS = [
    DATA_DIR / "wsj_headlines_1965_2014_zstd.parquet",
    DATA_DIR / "wsj_headlines_2015_2026_zstd.parquet",
]
NEWS_DATE_COL = "date"
NEWS_TEXT_COL = "headline"

# ── Standalone mode ───────────────────────────────────────────────────────────
# True = load corpus and embeddings from disk (no main pipeline needed)
# False = use news_df and article_embeddings already in memory
STANDALONE = False

# ── Stage 1: Structural filter ────────────────────────────────────────────────
MIN_WORDS = 4     # drop headlines with fewer words
MAX_WORDS = 40    # drop headlines with more words (likely concatenated ledes)

# ── Stage 2: Blocklist filter ─────────────────────────────────────────────────
# Explicit non-financial content categories.
# Only triggers on unambiguous terms — kept deliberately narrow to avoid
# false positives on financial headlines that happen to share vocabulary.
NON_FINANCIAL_BLOCKLIST = {
    # sports
    "quarterback","touchdown","inning","wicket","athlon","semifinals",
    "playoffs","championship","nfl","nba","mlb","nhl","fifa","olympics",
    "medalist","sprinter","golfer","racetrack","jockey",
    # entertainment
    "oscar","grammy","emmy","bafta","sundance","cannes","billboard",
    "sitcom","blockbuster","cameo","screenplay","cinematographer",
    "discography","vocalist","choreographer","broadway",
    # food / lifestyle
    "cookbook","sommelier","gastronomy","culinary","patisserie",
    "horoscope","crossword","sudoku","astrology","zodiac",
    # pure travel / leisure
    "sightseeing","snorkeling","parasailing","excursion",
}

# ── Stage 3: Anchor similarity filter ────────────────────────────────────────
# Covers macro, corporate actions, sector-specific, and historical vocabulary.
ANCHOR_PHRASES = [
    # macro / markets
    "corporate earnings quarterly results profit loss",
    "dividend payout shareholder return yield",
    "federal reserve interest rate monetary policy",
    "economic growth GDP inflation recession outlook",
    "stock market equity index trading shares",
    "merger acquisition takeover corporate deal",
    "employment jobs unemployment labor wages payroll",
    "oil energy commodity currency exchange rate dollar",
    "government spending fiscal deficit budget tax",
    "bank credit loan debt financial institution",
    # corporate actions — catches older operational headlines
    "company wins contract valued million billion",
    "acquisition merger deal corporate takeover",
    "chief executive officer appointed named president",
    "quarterly profit loss million billion revenue",
    "debt offering bonds debentures borrowing filing",
    "stock dividend shares offering capital raise",
]

# Threshold: headlines with max cosine similarity below this are dropped.
# 0.30 is intentionally conservative — only drops clearly non-financial content.
# Re-inspect and raise if too much noise remains after running.
ANCHOR_THRESHOLD = 0.30

# ── Anchor hash (auto-invalidates cache when phrases change) ──────────────────
ANCHOR_HASH = hashlib.md5(str(sorted(ANCHOR_PHRASES)).encode()).hexdigest()[:8]

print("Configuration loaded.")
print(f"Anchor hash: {ANCHOR_HASH}  ({len(ANCHOR_PHRASES)} phrases)")
print(f"Cache dir:   {CACHE_DIR}")

## 2. Load corpus and embeddings

In [ ]:
# ── Load news corpus ──────────────────────────────────────────────────────────
try:
    if STANDALONE: raise NameError
    _ = news_df
    print(f"Using news_df from main pipeline: {len(news_df):,} articles")
except NameError:
    frames = []
    for path in NEWS_PARQUETS:
        if not path.exists():
            print(f"WARNING: {path.name} not found — skipping.")
            continue
        df = pd.read_parquet(path)
        df = df.rename(columns={NEWS_DATE_COL: "date", NEWS_TEXT_COL: "text"})
        df = df[["date","text"]].dropna()
        df["date"] = pd.to_datetime(df["date"])
        frames.append(df)
        print(f"  Loaded {len(df):,} from {path.name}")
    news_df = (pd.concat(frames, ignore_index=True)
                 .drop_duplicates(subset=["date","text"])
                 .sort_values("date")
                 .reset_index(drop=True))
    print(f"Combined: {len(news_df):,} articles")

news_df["year"]       = news_df["date"].dt.year
news_df["decade"]     = (news_df["year"] // 10 * 10).astype(str) + "s"
news_df["word_count"] = news_df["text"].str.split().str.len()

# ── Load cached embeddings ────────────────────────────────────────────────────
try:
    if STANDALONE: raise NameError
    _ = article_embeddings
    print(f"Using article_embeddings from main pipeline: {article_embeddings.shape}")
except NameError:
    cache_key  = (f"news_emb_n{len(news_df)}"
                  f"_{news_df['date'].min():%Y%m%d}"
                  f"_{news_df['date'].max():%Y%m%d}.npy")
    cache_path = CACHE_DIR / cache_key
    if not cache_path.exists():
        raise FileNotFoundError(
            f"Embedding cache not found at {cache_path}.\n"
            "Run the main pipeline first to generate embeddings."
        )
    print(f"Loading embeddings from {cache_path}...")
    article_embeddings = np.load(cache_path)
    print(f"Loaded: {article_embeddings.shape}")

assert len(article_embeddings) == len(news_df),     f"Mismatch: {len(article_embeddings)} embeddings vs {len(news_df)} articles"
print("\nCorpus and embeddings aligned.")

## 3. Stage 1 — Structural filter

Remove headlines that are too short, too long, or contain no alphabetic
characters. These are wire alerts, section headers, corrections, and
concatenated lede text that carry no useful signal.

In [ ]:
def structural_filter(text):
    if not isinstance(text, str): return False
    words = text.split()
    if len(words) < MIN_WORDS: return False
    if len(words) > MAX_WORDS: return False
    if not any(c.isalpha() for c in text): return False
    return True

mask_structural = news_df["text"].apply(structural_filter)
n_structural    = mask_structural.sum()
n_drop_struct   = (~mask_structural).sum()

print(f"Structural filter:")
print(f"  Keep: {n_structural:,} ({n_structural/len(news_df):.1%})")
print(f"  Drop: {n_drop_struct:,} ({n_drop_struct/len(news_df):.1%})")

print(f"\nSample dropped headlines:")
for _, row in news_df[~mask_structural].sample(min(10, n_drop_struct),
                                                random_state=42).iterrows():
    print(f"  [{row['word_count']} words]  '{row['text']}'")

## 4. Stage 2 — Blocklist filter

Remove headlines containing unambiguous non-financial terms. The blocklist
is intentionally narrow — only terms that essentially never appear in
genuinely financial headlines — to minimise false positives.

In [ ]:
def blocklist_filter(text):
    tokens = set(re.findall(r"[a-zA-Z]+", text.lower()))
    return not bool(tokens & NON_FINANCIAL_BLOCKLIST)

# Apply on top of structural filter
mask_blocklist = mask_structural & news_df["text"].apply(blocklist_filter)
n_blocklist    = mask_blocklist.sum()
n_drop_block   = (mask_structural & ~mask_blocklist).sum()

print(f"Blocklist filter (applied after structural):")
print(f"  Keep: {n_blocklist:,} ({n_blocklist/len(news_df):.1%})")
print(f"  Drop: {n_drop_block:,} additional ({n_drop_block/n_structural:.1%} of structural-passed)")

print(f"\nSample dropped headlines:")
dropped_bl = news_df[mask_structural & ~mask_blocklist]
for _, row in dropped_bl.sample(min(15, len(dropped_bl)), random_state=42).iterrows():
    matched = [w for w in re.findall(r"[a-zA-Z]+", row["text"].lower())
               if w in NON_FINANCIAL_BLOCKLIST]
    print(f"  [{matched}]  {row['text']}")

## 5. Stage 3 — Anchor similarity filter

Compute cosine similarity between each article embedding and the anchor
phrase embeddings. Uses the cached article embeddings — no re-encoding.
Cache is keyed by corpus size + anchor hash so stale results are impossible.

In [ ]:
import torch

def embed_phrases(phrases, batch_size=32):
    """Embed anchor phrases using frozen FinBERT encoder."""
    try:
        _ = model, tokenizer, DEVICE
    except NameError:
        raise RuntimeError("Encoder not in memory. Run main pipeline or set STANDALONE=False.")
    all_emb = []
    model.eval()
    for i in range(0, len(phrases), batch_size):
        batch  = phrases[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=64)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs)
        all_emb.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(all_emb)


# ── Load or compute anchor similarity ─────────────────────────────────────────
anchor_cache_path = CACHE_DIR / f"anchor_sim_n{len(news_df)}_{ANCHOR_HASH}.npy"

if anchor_cache_path.exists():
    print(f"Loading cached anchor similarities from {anchor_cache_path}")
    anchor_sim_all = np.load(anchor_cache_path)
else:
    print("Computing anchor similarities (this runs once and is cached)...")
    print("Embedding anchor phrases...")
    anchor_emb  = embed_phrases(ANCHOR_PHRASES)
    anchor_norm = anchor_emb / (np.linalg.norm(anchor_emb, axis=1, keepdims=True) + 1e-9)
    emb_norm    = article_embeddings / (
        np.linalg.norm(article_embeddings, axis=1, keepdims=True) + 1e-9
    )
    sims = []
    chunk = 50_000
    for i in tqdm(range(0, len(emb_norm), chunk), desc="Similarity"):
        batch_sim = emb_norm[i:i+chunk] @ anchor_norm.T
        sims.append(batch_sim.max(axis=1))
    anchor_sim_all = np.concatenate(sims)
    np.save(anchor_cache_path, anchor_sim_all)
    print(f"Cached to {anchor_cache_path}")

news_df["anchor_sim"] = anchor_sim_all

print(f"\nAnchor similarity stats:")
print(f"  mean:   {anchor_sim_all.mean():.4f}")
print(f"  median: {np.median(anchor_sim_all):.4f}")
print(f"  p10:    {np.percentile(anchor_sim_all, 10):.4f}")
print(f"  p25:    {np.percentile(anchor_sim_all, 25):.4f}")
print(f"  p75:    {np.percentile(anchor_sim_all, 75):.4f}")

## 6. Threshold inspection

Inspect what the anchor filter keeps and drops before committing.
Change `INSPECT_THRESHOLD` and rerun to explore different cuts.

In [ ]:
INSPECT_THRESHOLD = ANCHOR_THRESHOLD   # change to explore

# Apply on top of structural + blocklist
mask_anchor = mask_blocklist & (news_df["anchor_sim"] >= INSPECT_THRESHOLD)
n_anchor    = mask_anchor.sum()
n_drop_anc  = (mask_blocklist & ~mask_anchor).sum()

kept    = news_df[mask_anchor]
dropped_anc = news_df[mask_blocklist & ~mask_anchor]

print(f"Anchor filter (threshold={INSPECT_THRESHOLD}, applied after blocklist):")
print(f"  Keep: {n_anchor:,} ({n_anchor/len(news_df):.1%} of corpus)")
print(f"  Drop: {n_drop_anc:,} additional")

# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f"Anchor similarity distribution (after structural + blocklist)",
             fontweight="bold")

ax = axes[0]
sim_passed = news_df.loc[mask_blocklist, "anchor_sim"]
ax.hist(sim_passed, bins=80, color="#2c5f8a", alpha=0.85, edgecolor="white")
ax.axvline(INSPECT_THRESHOLD, color="#c0392b", ls="--", lw=2,
           label=f"threshold={INSPECT_THRESHOLD}")
ax.set_xlabel("Max cosine similarity to any anchor")
ax.set_ylabel("Count"); ax.set_title("After structural + blocklist filters")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60, 0.70]
fracs = [(sim_passed >= t).mean() for t in thresholds]
ax.plot(thresholds, fracs, "o-", color="#2c5f8a", lw=2)
ax.set_xlabel("Threshold"); ax.set_ylabel("Fraction kept")
ax.set_title("Fraction kept vs threshold")
ax.axvline(INSPECT_THRESHOLD, color="#c0392b", ls="--", lw=1.5)
for t, f in zip(thresholds, fracs):
    ax.annotate(f"{f:.1%}", (t, f), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f"\n── 20 random DROPPED headlines (anchor filter) ──────────────────────")
for _, row in dropped_anc.sample(min(20, len(dropped_anc)), random_state=1).iterrows():
    print(f"  sim={row['anchor_sim']:.3f}  {row['text']}")

print(f"\n── 15 borderline KEPT headlines ─────────────────────────────────────")
borderline = kept.nsmallest(300, "anchor_sim").sample(min(15, len(kept)), random_state=1)
for _, row in borderline.iterrows():
    print(f"  sim={row['anchor_sim']:.3f}  {row['text']}")

## 7. Decade stability

Check that the filter doesn't disproportionately remove early-decade
headlines due to vocabulary drift between historical and modern financial
language.

In [ ]:
decades = sorted(news_df["decade"].unique())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Filter impact by decade", fontweight="bold")

# Drop rate by decade for each filter stage
ax = axes[0]
struct_drop = [((~mask_structural) & (news_df["decade"]==d)).sum() /
               (news_df["decade"]==d).sum() for d in decades]
block_drop  = [(mask_structural & ~mask_blocklist & (news_df["decade"]==d)).sum() /
               (news_df["decade"]==d).sum() for d in decades]
anchor_drop = [(mask_blocklist & ~mask_anchor & (news_df["decade"]==d)).sum() /
               (news_df["decade"]==d).sum() for d in decades]

x = range(len(decades))
w = 0.25
ax.bar([i-w for i in x],   struct_drop, w, label="Structural", color="#2c5f8a", alpha=0.85)
ax.bar([i   for i in x],   block_drop,  w, label="Blocklist",  color="#e07b39", alpha=0.85)
ax.bar([i+w for i in x],   anchor_drop, w, label="Anchor sim", color="#c0392b", alpha=0.85)
ax.set_xticks(list(x)); ax.set_xticklabels(decades, rotation=30, ha="right")
ax.set_ylabel("Fraction dropped"); ax.set_title("Drop rate by stage and decade")
ax.legend(); ax.grid(alpha=0.3, axis="y")

# Total kept fraction by decade
ax = axes[1]
total_kept = [(mask_anchor & (news_df["decade"]==d)).sum() /
              (news_df["decade"]==d).sum() for d in decades]
colors = ["#c0392b" if k < 0.50 else "#e07b39" if k < 0.70 else "#4a9e6b"
          for k in total_kept]
ax.bar(decades, total_kept, color=colors, alpha=0.85, edgecolor="white")
ax.axhline(0.50, color="red", ls="--", lw=1, alpha=0.6, label="50% kept")
ax.set_ylabel("Fraction of decade kept")
ax.set_title("Total kept fraction by decade")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

plt.tight_layout(); plt.show()

print("Keep rate by decade:")
for d in decades:
    n_dec   = (news_df["decade"]==d).sum()
    n_kept  = (mask_anchor & (news_df["decade"]==d)).sum()
    med_sim = news_df[news_df["decade"]==d]["anchor_sim"].median()
    print(f"  {d}  kept={n_kept/n_dec:.1%}  median_sim={med_sim:.4f}  n={n_dec:,}")

## 8. Wave coverage check

Ensure no survey wave drops to zero or near-zero articles after all three
filter stages. Requires `all_results` from the main pipeline.

In [ ]:
AGG_WINDOW_DAYS     = 30
MIN_ARTICLES_WARN   = 5

try:
    _ = all_results
    HAS_RESULTS = True
except NameError:
    HAS_RESULTS = False
    print("all_results not available — run main pipeline first.")

if HAS_RESULTS:
    for series_name, res in all_results.items():
        waves = res["waves_df"]
        print(f"\n{'='*60}")
        print(f"  {series_name}  ({len(waves)} waves)")
        print(f"{'='*60}")
        print(f"{'Wave':<12} {'Raw':>6} {'S1':>6} {'S2':>6} {'S3':>6}")
        print(f"  (S1=structural, S2=+blocklist, S3=+anchor@{ANCHOR_THRESHOLD})")
        print("─" * 45)

        zero_waves = []
        for _, wave in waves.iterrows():
            wd   = wave["wave_date"]
            m    = (
                (news_df["date"] >= wd - pd.Timedelta(days=AGG_WINDOW_DAYS)) &
                (news_df["date"] <  wd)
            )
            raw  = m.sum()
            s1   = (m & mask_structural).sum()
            s2   = (m & mask_blocklist).sum()
            s3   = (m & mask_anchor).sum()
            flag = " ⚠" if s3 < MIN_ARTICLES_WARN else ""
            print(f"{wd.strftime('%Y-%m'):>12} {raw:>6} {s1:>6} {s2:>6} {s3:>6}{flag}")
            if s3 == 0:
                zero_waves.append(wd.strftime('%Y-%m'))

        if zero_waves:
            print(f"\n  ⚠ Zero-article waves: {zero_waves}")
        else:
            print(f"\n  No zero-article waves.")

## 9. Save filtered index

Save the boolean mask and the filtered article indices to disk.
The main pipeline loads this to apply the filter during wave aggregation
without re-running the filter each time.

In [ ]:
# Final combined mask
filter_mask = mask_anchor.values   # boolean array, length = len(news_df)

# Summary
n_raw    = len(news_df)
n_final  = filter_mask.sum()
n_drop   = n_raw - n_final

print(f"Filter summary:")
print(f"  Raw corpus:          {n_raw:>10,}")
print(f"  After structural:    {mask_structural.sum():>10,}  "
      f"(-{(~mask_structural).sum():,})")
print(f"  After blocklist:     {mask_blocklist.sum():>10,}  "
      f"(-{(mask_structural & ~mask_blocklist).sum():,})")
print(f"  After anchor sim:    {n_final:>10,}  "
      f"(-{(mask_blocklist & ~mask_anchor).sum():,})")
print(f"  Total drop:          {n_drop:>10,}  ({n_drop/n_raw:.1%})")

# Save
filter_key  = (f"filter_mask_n{n_raw}"
               f"_{news_df['date'].min():%Y%m%d}"
               f"_{news_df['date'].max():%Y%m%d}"
               f"_anc{ANCHOR_HASH}"
               f"_t{str(ANCHOR_THRESHOLD).replace('.','')}.npy")
filter_path = CACHE_DIR / filter_key

np.save(filter_path, filter_mask)
print(f"\nFilter mask saved to: {filter_path}")
print(f"\nTo apply in main pipeline, add to aggregate_to_waves:")
print(f'  FILTER_MASK = np.load("{filter_path}")')
print(f"  # then inside the wave loop, after selecting idx:")
print(f"  idx = idx[FILTER_MASK[idx]]")

## 10. Main pipeline integration

The filter is applied in `aggregate_to_waves` by pre-filtering article
indices before the sentiment filter and recency weighting. Add the
following to the main pipeline.

In [ ]:
# ── Copy this block into the main pipeline ────────────────────────────────────

# In config cell (cell 5), add:
integration_config = """
# ── Headline filter (from headline_cleanup_pipeline.ipynb) ────────────────────
# Path to the saved filter mask — update after re-running cleanup pipeline.
FILTER_MASK_PATH = CACHE_DIR / "" + filter_key + ""
FILTER_MASK      = np.load(FILTER_MASK_PATH) if FILTER_MASK_PATH.exists() else None
if FILTER_MASK is not None:
    print(f"Headline filter loaded: {FILTER_MASK.sum():,} articles kept "
          f"({FILTER_MASK.mean():.1%} of corpus)")
else:
    print("WARNING: headline filter not found — running without it.")
"""

# In aggregate_to_waves, add immediately after the NaN embedding filter:
integration_agg = """
        # ── Headline relevance filter ─────────────────────────────────────────
        if FILTER_MASK is not None:
            keep = FILTER_MASK[idx]
            if keep.sum() == 0:
                # All filtered — fall back to keeping all articles for this wave
                pass
            else:
                emb, idx = emb[keep], idx[keep]
"""

print("Integration config snippet:")
print(integration_config)
print("Integration aggregate_to_waves snippet:")
print(integration_agg)